# 🚀 AMVspace (AMVerge على Colab)

1. **Runtime ← Change runtime type ← T4 GPU**
2. **Runtime ← Run all** (التشغيل الأول يبني الواجهة ويستغرق عدة دقائق)
3. انتظر ظهور رابط `https://….trycloudflare.com` في آخر خلية، وافتحه في **Chrome** على هاتفك.
4. من قائمة Chrome اختر **Add to Home screen** لتصير AMVspace تطبيقاً على شاشتك.

> مبني على AMVerge (GPL-3.0). الكود: github.com/hmar43661-droid/AMVspace


In [ ]:
!pip -q install fastapi "uvicorn[standard]" "amverge[edge,ml]"
!if ! command -v node >/dev/null 2>&1 || [ "$(node -p 'process.versions.node.split(".")[0]')" -lt 18 ]; then curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1; apt-get install -y nodejs >/dev/null 2>&1; fi; node -v


In [ ]:
import os, subprocess, zipfile, shutil
# إن رفعت amvspace_web.zip يدوياً إلى /content سيُستخدم، وإلا يُحمَّل من مستودعك
ZIP_URL = "https://github.com/hmar43661-droid/AMVspace/raw/main/amvspace_web.zip"
if not os.path.exists("/content/amvspace_web.zip"):
    subprocess.run(["wget", "-q", "-O", "/content/amvspace_web.zip", ZIP_URL], check=True)
shutil.rmtree("/content/src_web", ignore_errors=True)
zipfile.ZipFile("/content/amvspace_web.zip").extractall("/content/src_web")
os.makedirs("/content/amv", exist_ok=True)
shutil.copy("/content/src_web/server/server.py", "/content/amv/server.py")
print("OK: source downloaded")


In [ ]:
%%bash
set -e
cd /content/src_web/web
npm ci --no-audit --no-fund 2>&1 | tail -3
npx vite build --config vite.web.config.ts 2>&1 | tail -15
rm -rf /content/amv/ui && mkdir -p /content/amv/ui && cp -r dist/. /content/amv/ui/
ls /content/amv/ui | head


In [ ]:
import subprocess, sys, time, re, threading, requests, os, torch
if not torch.cuda.is_available():
    print("⚠️ لم يتم تفعيل GPU: Runtime ← Change runtime type ← T4 GPU")
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared")
srv = subprocess.Popen([sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"], cwd="/content/amv")
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:8000/health", timeout=2)
        break
    except Exception:
        time.sleep(1)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                       stderr=subprocess.PIPE, text=True)
url = None
for line in tun.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tun.stderr], daemon=True).start()
print("=" * 60)
print("✅ افتح هذا الرابط في Chrome على هاتفك:")
print(url)
print("=" * 60)
print("اترك هذه الخلية تعمل ولا تغلق Colab أثناء الاستخدام.")
while True:
    time.sleep(30)
